### Test unitarios del pipeline completo de contabilidad mensual

In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path("../src").resolve()))
from extract import extraer_facturas_excel

In [ ]:
## Test de extract.py
archivo = Path("../data/raw/contabilidad/CONTABILIDAD 2026-01_ISLA.xlsx")

compras, ventas = extraer_facturas_excel(archivo)

print(compras.head())
print(ventas.head())

print(compras.shape)
print(ventas.shape)

In [ ]:
## Test de transform.py
from pathlib import Path
import sys
sys.path.append(str(Path("../src").resolve()))
from extract import extraer_facturas_excel
from transform import transformar_facturas
from tests import test_facturas

archivo = Path("../data/raw/contabilidad/CONTABILIDAD 2026-01_ISLA.xlsx")

compras, ventas = extraer_facturas_excel(archivo)
# Transformación
compras_transformadas = transformar_facturas(compras, "compras")

ventas_transformadas = transformar_facturas(ventas, "ventas")

print(compras_transformadas.head())
print(ventas_transformadas.head())

In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path("../src").resolve()))
from extract import extraer_facturas_excel
from transform import transformar_facturas
from tests import test_facturas


archivo = Path("../data/raw/contabilidad/CONTABILIDAD 2026-01_ISLA.xlsx")

compras, ventas = extraer_facturas_excel(archivo)

compras_transformadas = transformar_facturas(compras, "compras")
ventas_transformadas = transformar_facturas(ventas, "ventas")

test_facturas(compras_transformadas, "compras")

test_facturas(ventas_transformadas, "ventas")

VALIDANDO DATAFRAME
 DataFrame no vacío.
Columnas correctas de compras y/o ventas.
Fechas válidas.
Tipos de datos numericos correctos.
Tipos de datos string correctos.
Fechas dentro del rango esperado.
Sin valores nulos críticos.
Trazabilidad correcta.
TODAS LAS VALIDACIONES FUERON EXITOSAS
VALIDANDO DATAFRAME
 DataFrame no vacío.
Columnas correctas de compras y/o ventas.
Fechas válidas.
Tipos de datos numericos correctos.
Tipos de datos string correctos.
Fechas dentro del rango esperado.
Sin valores nulos críticos.
Trazabilidad correcta.
TODAS LAS VALIDACIONES FUERON EXITOSAS


In [ ]:
#Hubo problema con el tipo de dato, revisamos que todas sean str
for col in ['tipo_comprobante','punto_venta', 'numero_comprobante', 'razon_social', 'cuit', 'archivo_origen','condicion','concepto','forma_pago']:
    print(col, compras_transformadas[col].dtype)

tipo_comprobante string
punto_venta int64
numero_comprobante int64
razon_social string
cuit string
archivo_origen str
condicion string
concepto string
forma_pago string


In [ ]:
# Problema porque no coincidia el total vs la suma de todos los conceptos
df = compras_transformadas.copy()

total_calculado = (df["neto_gravado"]  + df["iva_21"] + df["iva_10_5"] + df["iva_27"] + df["iva_3"] + df["percepcion_iibb"] +
    df["percepcion_municipal"] + df["no_gravado_exento"])

df["diferencia"] = abs(total_calculado - df["total"])
# Vemos cual es esa fila
df[df["diferencia"] >= 1][["fecha","razon_social", "neto_gravado","iva_21", "iva_10_5", "iva_27","iva_3","percepcion_iibb",
                           "percepcion_municipal", "no_gravado_exento", "total", "diferencia"]]

In [ ]:
#Habia datos duplicdos
duplicados = compras_transformadas[compras_transformadas.duplicated
                                   (subset=["tipo_comprobante", "punto_venta", "numero_comprobante"], keep=False)]
duplicados

In [ ]:
#Habia problemas con el formato de cuantos numeros y el tipo de formato del dato de cuit
ventas_transformadas["cuit"].dropna().apply(lambda x: (x, len(str(x)))).drop_duplicates()

90    (30707620192.0, 13)
Name: cuit, dtype: object